In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

85

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


The ID becomes the label in our ground truth dataset. 

This is why every record needs a stable ID. If you can't uniquely identify a document, you can't tell whether search retrieved the right one. When you build your own evaluation set, assign an ID to each record in your knowledge base first.

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

Until now we called responses.create and read response.output_text. For structured output we switch to responses.parse and pass text_format=Questions, which tells the API to return our class instead of free text.

### Below is an updated version of the course, using groq instad

In [10]:
response = openai_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "developer", "content": data_gen_instructions},
        {"role": "user", "content": user_prompt}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "Questions_schema", # Changed "Questions schema" to "Questions_schema"
            "schema": Questions.model_json_schema()
        }
    }
)

In [11]:
raw_result = json.loads(response.choices[0].message.content or "{}")
result = Questions.model_validate(raw_result)
print(result.model_dump_json(indent=2))

{
  "questions": [
    "I just found out about the LLM Zoomcamp, can I still enroll?",
    "Is it possible to join the course now that it's already started?",
    "Can I still sign up for the course and still get the certificate?",
    "Do I have to miss the deadline to get a certificate if I join late?",
    "If I join late, can I still submit the project for certification?"
  ]
}


In [12]:
from groq_evaluation_utils import llm_structured

In [13]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Is it possible to enroll in the LLM Zoomcamp now that I just found out about it?', 'Can I still get a certificate if I sign up late?', 'Do I have to finish the project before the submission deadline to earn the certificate?', 'What are the requirements for receiving a certificate if I join the course late?', 'Are there any restrictions on joining the course after it has already started?']


# Note: llama-3.3-70b-versatile does not handle json format outputs, so use openai/gpt-oss-120b

In [14]:
usage

CompletionUsage(completion_tokens=626, prompt_tokens=332, total_tokens=958, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=527, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.633849614, prompt_time=0.015539611, completion_time=1.31996022, total_time=1.335499831)

In [15]:
# Assuming 'usage' is your CompletionUsage object
print(f"Total Tokens: {usage.total_tokens}")
print(f"Completion Tokens: {usage.completion_tokens}")
print(f"Prompt Tokens: {usage.prompt_tokens}")

Total Tokens: 958
Completion Tokens: 626
Prompt Tokens: 332


In [16]:
# or:
# Convert the entire object to a dictionary
usage_dict = usage.model_dump()

# Access values like a dictionary
print(usage_dict["total_tokens"])

958


In [17]:
from groq_evaluation_utils import calc_price

In [18]:
# reload groq_evaluation_utils to get the updated calc_price function
import importlib
import groq_evaluation_utils
importlib.reload(groq_evaluation_utils)

<module 'groq_evaluation_utils' from '/Users/nhubert/Documents/llm-zoomcamp-code/04-orchestration/groq_evaluation_utils.py'>

In [19]:
from groq_evaluation_utils import calc_price

In [20]:
cost = calc_price(usage)

print(cost.total_cost)   # Works!
print(cost.input_cost)   # Works!

0.003066
0.000249


In [21]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it possible to enroll in the LLM Zoomcamp now that I just found out about it?',
  'document': '74eb249bbf'},
 {'question': 'Can I still get a certificate if I sign up late?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to finish the project before the submission deadline to earn the certificate?',
  'document': '74eb249bbf'},
 {'question': 'What are the requirements for receiving a certificate if I join the course late?',
  'document': '74eb249bbf'},
 {'question': 'Are there any restrictions on joining the course after it has already started?',
  'document': '74eb249bbf'}]

In [22]:
from groq_evaluation_utils import llm_structured_retry
importlib.reload(groq_evaluation_utils)

<module 'groq_evaluation_utils' from '/Users/nhubert/Documents/llm-zoomcamp-code/04-orchestration/groq_evaluation_utils.py'>

In [23]:
from groq_evaluation_utils import llm_structured_retry

In [24]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [25]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

This works, but it runs one LLM call after another. Running it for all documents this way would take too long. ==> **parallel processing**

## Parallel processing

Running the calls one after another wastes most of the time waiting on the network. Each request just sits there until OpenAI responds, so we can fire several at once and wait on them together. We process the documents in parallel and track progress while the requests run.

One caution: don't open too many connections at once, or you'll hit the provider's rate limits. Five or six workers is a safe default here.

In [26]:
from concurrent.futures import ThreadPoolExecutor
from groq_evaluation_utils import map_progress

In [27]:
documents = documents[:9]  # Limit to 9 documents for testing, because I use Groq free tier

In [28]:
with ThreadPoolExecutor(max_workers=3) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/9 [00:00<?, ?it/s]

In [29]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

45

In [30]:
from groq_evaluation_utils import calc_price

# Simplified calculation using a generator expression
total_cost = sum(calc_price(usage).total_cost for usage in usages)

print(total_cost)

0.01999875


In [31]:
from groq_evaluation_utils import calc_total_price

calc_total_price(usages)

0.01999875

In [32]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [33]:
df_ground_truth.to_csv("ground_truth-new.csv", index=False)

# 4 Search Evaluation

In [34]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [35]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [36]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [37]:
q = ground_truth[0]
q

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [38]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [39]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
acf8fa5356 == 74eb249bbf: False


In [40]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [41]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [42]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Is it okay to join the course late if I just found it now?


[1, 0, 0, 0, 0]

The correct document was the first search result.

In [44]:
q = ground_truth[14]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Should I type my questions in the live chat during Office Hours, or use something else?


[0, 1, 0, 0, 0]

In [45]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [46]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [47]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

Each entry in relevance_total_text is a relevance list. This is enough to check that the function works before we run it for the full dataset.

Next, make the relevance functions generic. We start with text search, but later we may want to evaluate vector search, hybrid search, or another retrieval method. The relevance logic is the same. Only the search function changes.

In [48]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [49]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [50]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

Now we can represent search results as relevance lists. In the next lesson, we'll turn these lists into metrics: Hit Rate and MRR.

# 5 Search Evaluation Metrics

In [51]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In our setup, each query has one correct document, so Hit Rate and Recall@k mean the same thing.

In [52]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

hit_rate(example)

0.9333333333333333

Hit Rate tells us if we found the right document, but not where it was.

MRR also considers the position.

For each query, the score is based on the rank of the first correct document:

position 1: score is 1.0
position 2: score is 0.5
position 3: score is 0.333
not found: score is 0

In [53]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [54]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [55]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6734177215189874, 'mrr': 0.5895780590717299}

# 6 Search Parameter Tuning

In [56]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [57]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.7139240506329114, 'mrr': 0.620084388185654}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.7139240506329114, 'mrr': 0.6206751054852321}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.6734177215189874, 'mrr': 0.5895780590717299}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.660759493670886, 'mrr': 0.5727848101265822}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.6455696202531646, 'mrr': 0.560168776371308}


Increasing the question boost makes the metrics worse, not better. The best value here is 1.0, no boost at all. That's already the opposite of what the intuition predicted.

But this is only one parameter. We can also tune answer and section together with question.

Define a search function with all three boosts:



In [58]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [59]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/395 [00:00<?, ?it/s]

In [60]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
35,5.0,10.0,0.5,0.767089,0.675865
19,2.0,4.0,0.2,0.767089,0.675865
3,1.0,2.0,0.1,0.767089,0.675865
34,5.0,10.0,0.2,0.767089,0.675654
18,2.0,4.0,0.1,0.767089,0.675654
33,5.0,10.0,0.1,0.764557,0.675021
4,1.0,2.0,0.2,0.764557,0.671646
20,2.0,4.0,0.5,0.759494,0.671224
7,1.0,4.0,0.2,0.759494,0.669958
6,1.0,4.0,0.1,0.759494,0.667764


In [62]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Top-K tradeoffs
We return 5 results from search. **Increasing top-K to 10 would improve hit rate because there are more chances to find the correct document**. **But more results means more context sent to the LLM. That costs more and makes it harder for the model to identify what is relevant**. Five results is a reasonable default for short FAQ-style documents.

# 7 RAG and Agent Evaluation

RAG evaluation checks the whole flow together.

This includes:

* search
* prompt
* LLM

If the final answer is bad, the problem can come from any of these steps.

## LLM as a judge

For RAG and agent evaluation, we compare the generated answer with the original answer. The generated answer won't use the same words as the original. It's a generative model, so the phrasing will be different even when the meaning is the same.

This is why we use another LLM to do the comparison. We show the judge the question, the original answer, and the generated answer. Then we ask it to decide if they are semantically equivalent.

This approach is called LLM-as-a-judge. The evaluating LLM is the judge. It classifies each answer as good or bad and explains its reasoning. Asking the judge to explain why it made a decision generally produces better classifications than asking for just the verdict.

# 8 Generating RAG Answers

In [63]:
import pandas as pd
from ingest import load_faq_data, build_index

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [64]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

## Running RAG

In [65]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [66]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

For each question, RAGBase searches the FAQ, builds a prompt with the retrieved context, and asks the LLM to answer. We save the answer so the next lesson can judge it.

Run RAG for one question:

In [67]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

"According to the provided context, the answer to your question is: **Yes**, it is okay to join the course late if you just found it now. However, if you want to receive a certificate, you need to submit your project while they're still accepting submissions."

In [68]:
assistant.total_cost()

0.00057375

In [69]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [70]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': "According to the provided context, the answer to your question is: **Yes**, it is okay to join the course late if you just found it now. However, if you want to receive a certificate, you need to submit your project while they're still accepting submissions.",
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [71]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [72]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'According to the General Course-Related Questions section, the answer is: **Yes**, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [73]:
assistant.reset_usage()

In [74]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [75]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [76]:
assistant.total_cost()

0.0

In [77]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

# 9 LLM as a Judge

For offline evaluation, we have three things:

- the original FAQ answer
- the question generated from that answer
- the answer generated by our RAG pipeline

An LLM judge is another LLM call that compares these three pieces. We
ask it whether the RAG answer recovers the same information as the
original answer.

It can also explain why an answer is bad.

For example:

- the retrieved document might be wrong
- the answer might miss the key point
- the model might say that it doesn't know

This approach is useful when exact text matching is too strict. The RAG
answer doesn't need to copy the FAQ answer word for word. It needs to
answer the question with the same key information.

This evaluates the full RAG flow in one pass:

- search: did we retrieve context that contains the answer?
- prompt: did we give the model enough context to answer?
- LLM: did the model produce a useful answer from that context?

If the judge marks an answer as bad, we still need to look at the
example. The judge tells us where to investigate. It doesn't replace
reading the failing cases.

In [93]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

**In production, we usually don't have that original answer for real user questions. There we can still use an LLM judge. The prompt has to judge only the question and the generated answer.** In this lesson, we use the stronger offline setup.

In [94]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [95]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [96]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [97]:
aqa_judge_prompt

'Question:\n{question}\n\nOriginal Answer (ground truth):\n{answer_orig}\n\nAI Answer:\n{answer_llm}'

In [110]:
import importlib
import evaluation_utils

importlib.reload(evaluation_utils)

from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

In [111]:
rec = answers[0]

In [112]:
rec

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [113]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [114]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer conveys the same information as the original: it confirms that joining late is allowed and notes that to get a certificate the project must be submitted while submissions are still open. No key points are missing or incorrect.', score='good')

In [115]:
calc_price(usage)

{'input_cost': 0.00032175, 'output_cost': 0.0006975, 'total_cost': 0.00101925}

In [117]:
def evaluate_aqa(question, answer_orig, answer_llm, model="openai/gpt-oss-120b"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [118]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer conveys the same key information as the original: joining late is allowed, and obtaining a certificate requires submitting the project before the submission deadline. No essential details are missing or incorrect.', score='good')

In [119]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [121]:
answers = answers[:12] # cannot affort much with Groq free tier, so we limit to 12 records for testing

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/12 [00:00<?, ?it/s]

In [ ]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [ ]:
df_eval = pd.DataFrame(evaluations)

In [ ]:
calc_total_price(usages)

In [ ]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

In [ ]:
df_eval[df_eval["score"] == "bad"].head()

These rows are often the most useful part of the evaluation. They can show that search retrieved the wrong document. They can also show that the answer is too generic. Sometimes the RAG pipeline says that it doesn't know even though the FAQ had the answer.